In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('history_traduzido.csv')

features_categoricas = [
    'genero', 'possui_companheiro', 'possui_dependentes', 'servico_telefonico', 
    'multiplas_linhas_telefonicas', 'servico_internet', 'seguranca_online', 
    'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 
    'streaming_filmes', 'tipo_contrato', 'faturamento_digital', 'metodo_pagamento'
]
features_numericas = ['idoso', 'tempo_permanencia', 'cobranca_mensal', 'cobranca_total_acumulada', 'renda_mensal']

encoder = LabelEncoder()
for col in features_categoricas:
    df[col] = encoder.fit_transform(df[col].astype(str))

df['cancelamento_servico'] = df['cancelamento_servico'].map({'Yes': 1, 'No': 0})

X_tabular = df[features_categoricas + features_numericas].values

modelo_nlp = SentenceTransformer('all-MiniLM-L6-v2')

feedbacks = df['feedback_cliente'].fillna("").tolist()
X_texto = modelo_nlp.encode(feedbacks, show_progress_bar=True)

X_final = np.hstack((X_tabular, X_texto))
y = df['cancelamento_servico'].values

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42, stratify=y)

modelo_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

previsoes = modelo_xgb.predict(X_test)
print(classification_report(y_test, previsoes, target_names=['Retido (0)', 'Cancelou (1)']))

c:\Users\Lois\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lois\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 177/177 [02:42<00:00,  1.09it/s]


              precision    recall  f1-score   support

  Retido (0)       0.99      1.00      0.99       826
Cancelou (1)       0.99      0.96      0.97       301

    accuracy                           0.99      1127
   macro avg       0.99      0.98      0.98      1127
weighted avg       0.99      0.99      0.99      1127



Support (1127): O modelo separou 1.127 clientes para a prova final (o conjunto de teste). Desses, 826 não cancelaram (Retido) e 301 cancelaram o serviço (Cancelou).

Precision (0.99): De todos os clientes que o seu algoritmo apontou o dedo e disse "vai cancelar", ele acertou 99% das vezes. A precisão avalia a qualidade do alerta.

Recall (0.96): Dos 301 clientes que de fato foram embora, o algoritmo conseguiu rastrear e identificar 96% deles. Os 4% restantes escaparam do radar (falsos negativos).

F1-Score (0.97): É a média harmônica entre a precisão e o recall, indicando um modelo quase perfeito. No mundo real da análise de dados, um F1-Score de 0.97 em churn é o primeiro sintoma de que o modelo tem informações privilegiadas.

In [4]:
import pandas as pd

df_textos = pd.read_csv('history_traduzido.csv')

# clientes que cancelaram (o CSV mantém o 'Yes')
clientes_que_cancelaram = df_textos[df_textos['cancelamento_servico'] == 'Yes']

# 10 feedbacks completos para análise qualitativa
for index, texto in enumerate(clientes_que_cancelaram['feedback_cliente'].head(10)):
    print(f"Texto {index + 1}: {texto}\n")

Texto 1: I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.

Texto 2: I have been a customer with this company for 70 months and initially signed up for a two-year contract. I have been using their fiber optic internet service and have been satisfied with the speed and reliability. However, I recently decided to churn because I found a better deal with another provider. The monthly charges of $98.3 were reasonable for the service I received. I appreciated the convenience of automatic bank transfers for payment. Overall, my experience with this company was positive, but I ultimately

Vazamento de Dados Data Leakage
Por conta do cancel, churn etc, ele simplesmente leu a palavra "cancel" no texto e cravou que o cliente cancelou.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import plotly.graph_objects as go

df = pd.read_csv('history_traduzido.csv')

# EXCLUINDO o 'feedback_cliente'
features_categoricas = [
    'genero', 'possui_companheiro', 'possui_dependentes', 'servico_telefonico', 
    'multiplas_linhas_telefonicas', 'servico_internet', 'seguranca_online', 
    'backup_online', 'protecao_dispositivos', 'suporte_tecnico', 'streaming_tv', 
    'streaming_filmes', 'tipo_contrato', 'faturamento_digital', 'metodo_pagamento'
]
features_numericas = [
    'idoso', 'tempo_permanencia', 'cobranca_mensal', 
    'cobranca_total_acumulada', 'renda_mensal'
]

#  Transformar texto em números 
encoder = LabelEncoder()
for col in features_categoricas:
    df[col] = encoder.fit_transform(df[col].astype(str))

# Converter a coluna alvo para 1 (Cancelou) e 0 (Retido)
df['cancelamento_servico'] = df['cancelamento_servico'].map({'Yes': 1, 'No': 0})

X = df[features_categoricas + features_numericas]
y = df['cancelamento_servico']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#  XGBoost apenas com os dados tabulares
modelo_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

# acurácia REAL do modelo
previsoes = modelo_xgb.predict(X_test)
print("--- RELATÓRIO DE PREVISÃO DE CHURN (SEM TEXTO) ---")
print(classification_report(y_test, previsoes, target_names=['Retido (0)', 'Cancelou (1)']))

#  Visualizar as variáveis de maior risco
importancias = modelo_xgb.feature_importances_
nomes_features = features_categoricas + features_numericas

df_importancia = pd.DataFrame({'Feature': nomes_features, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=True)

# Top 10 para não poluir o gráfico
top_10 = df_importancia.tail(10)

fig = go.Figure(go.Bar(
    x=top_10['Importancia'],
    y=top_10['Feature'],
    orientation='h',
    marker=dict(color='royalblue')
))

fig.update_layout(
    title='Top 10 Variáveis que mais indicam risco de Churn',
    xaxis_title='Importância no Modelo',
    yaxis_title='Atributos do Cliente',
    height=500,
    template='plotly_white'
)

fig.show()

--- RELATÓRIO DE PREVISÃO DE CHURN (SEM TEXTO) ---
              precision    recall  f1-score   support

  Retido (0)       0.83      0.89      0.86       826
Cancelou (1)       0.62      0.50      0.55       301

    accuracy                           0.78      1127
   macro avg       0.72      0.69      0.71      1127
weighted avg       0.77      0.78      0.78      1127



In [7]:
!pip install --upgrade nbformat

   ---------------------------------------- 0.0/79.8 kB ? eta -:--:--
   ---------------------------------------- 79.8/79.8 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
   ---------------------------------------- 90.6/90.6 kB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/67.5 kB ? eta -:--:--
   ---------------------------------------- 67.5/67.5 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/226.0 kB ? eta -:--:--
   --------------------------------------- 226.0/226.0 kB 13.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
